# Stage 3: Implementing an RAG System for Question Answering

## Libraries

In [ ]:
!pip install pandas numpy seaborn matplotlib

In [21]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter
from openai import OpenAI
import json
import time
from google.colab import userdata



## Environment setup

In [ ]:
ENV = "colab"

In [ ]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In your Google Drive, create a folder "CLT"
In the folder CLT, create another folder `"stage_3"` and upload the csv `clean_title_content.csv`.

# 1. Data Loading


In [ ]:
if ENV == "colab":
    norm_path = '/content/drive/My Drive/CLT/stage_3/clean_title_content.csv'
else:
    norm_path = 'data/stage_3/clean_title_content.csv'

df = pd.read_csv(norm_path, encoding='utf-8', sep=';')

# Restore list columns serialized as strings by CSV
list_cols = ['tags']   # add more columns here if needed, e.g. ['tags', 'entities']

for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: ast.literal_eval(x) if pd.notna(x) else []
        )


print("Loaded shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3))

Loaded shape: (16518, 6)

Columns:
['date', 'domain', 'url', 'tags', 'title', 'content']

First 3 rows:
       date          domain  \
0  11.09.24  therobotreport   
1  11.09.24   geeksforgeeks   
2  11.09.24        newatlas   

                                                 url  \
0  https://www.therobotreport.com/ricoh-provides-...   
1  https://www.geeksforgeeks.org/ai-design-trends...   
2  https://newatlas.com/technology/llm-novel-rese...   

                                                tags  \
0                  [robotics, video, bostondynamics]   
1  [personalisation, naturallanguageprocessing, c...   
2  [largelanguagemodel, chatgpt, anthropic, claud...   

                                               title  \
0  ricoh to provide customer support for agility ...   
1  ai design trends to watch in 2025 from photore...   
2  ais generate more novel and exciting research ...   

                                             content  
0  the digit humanoid could work in distr

In the following code block we are searching for mostfrequent tags

In [ ]:
# Flatten list of tags
all_tags = [tag.lower() for tags in df['tags'] for tag in tags]

# Count frequencies
tag_counter = Counter(all_tags)

# Convert to DataFrame
tag_df = (
    pd.DataFrame(tag_counter.items(), columns=['tag', 'count'])
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

print("Number of unique tags:", len(tag_df))
print(tag_df.head(20))

Number of unique tags: 956
                          tag  count
0                generativeai   4201
1                       video   3257
2          largelanguagemodel   3106
3             personalisation   2594
4                    planning   2461
5                      openai   2386
6            conversationalai   2267
7          militaryanddefense   2171
8                    robotics   2155
9                     chatgpt   2110
10             accountability   1731
11                  reasoning   1656
12                 finetuning   1285
13          autonomousdriving   1245
14                      audio   1182
15           quantumcomputing   1073
16               deeplearning   1071
17                        gpu   1046
18                     gemini   1042
19  naturallanguageprocessing   1025


In [ ]:
# =========================
# 2) Topic groups from your tags
# =========================
topic_groups = {
    "generative_ai": [
        "generativeai", "largelanguagemodel", "chatgpt", "openai", "gemini"
    ],
    "nlp": [
        "naturallanguageprocessing", "conversationalai"
    ],
    "robotics_autonomy": [
        "robotics", "autonomousdriving"
    ],
    "ethics_governance": [
        "accountability", "militaryanddefense"
    ],
    "technical_ai": [
        "deeplearning", "finetuning", "gpu"
    ],
    "reasoning_planning": [
        "reasoning", "planning"
    ],
    "applications": [
        "personalisation", "video", "audio"
    ],
    "emerging_tech": [
        "quantumcomputing"
    ]
}

# =========================
# 3) Assign each document to one or more topic groups
# =========================
def assign_topic_groups(tags):
    tags_lower = {str(t).strip().lower() for t in tags}
    matched = []
    for group, group_tags in topic_groups.items():
        if any(tag in tags_lower for tag in group_tags):
            matched.append(group)
    return matched

df['topic_groups'] = df['tags'].apply(assign_topic_groups)

# Keep only rows that match at least one topic group
df_topics = df[df['topic_groups'].apply(len) > 0].copy()

print("Rows with matched topic groups:", df_topics.shape[0])


Rows with matched topic groups: 14510


In [ ]:
# =========================
# 4) Passage extraction from content
#    - prefer paragraphs
#    - fallback to sentence windows
# =========================
def normalize_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def split_paragraphs(text):
    # split on blank lines or paragraph-like breaks
    raw_parts = re.split(r'\n\s*\n|\r\n\s*\r\n', str(text))
    parts = [re.sub(r'\s+', ' ', p).strip() for p in raw_parts]
    return [p for p in parts if p]

def split_sentences(text):
    # simple sentence splitter
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def score_passage(passage):
    """
    Higher score = better candidate for QA generation.
    Rewards informative, self-contained passages.
    """
    p = str(passage)
    word_count = len(p.split())

    # base length preference: target around 80-220 words
    if word_count < 40:
        length_score = -100
    elif 80 <= word_count <= 220:
        length_score = 30
    elif 40 <= word_count < 80 or 220 < word_count <= 300:
        length_score = 15
    else:
        length_score = 0

    # reward informative features
    feature_score = 0
    if re.search(r'\b(ai|artificial intelligence|model|models|robot|policy|regulation|ethics|openai|google|microsoft|meta|nvidia|chatgpt|llm)\b', p, re.I):
        feature_score += 10
    if re.search(r'\b\d{4}\b', p):  # year mention
        feature_score += 3
    if re.search(r'\b(because|however|while|although|but|therefore|raises|improves|reduces|enables|compared to)\b', p, re.I):
        feature_score += 6

    # penalize noisy passages
    noise_score = 0
    if p.count('|') > 2:
        noise_score -= 10
    if len(re.findall(r'[^\w\s.,;:!?()\'"-]', p)) > 20:
        noise_score -= 10

    return length_score + feature_score + noise_score

def extract_best_passage(text, min_words=60, max_words=220):
    """
    Try paragraph-based extraction first.
    If no good paragraph, fallback to sentence-window extraction.
    """
    text = str(text).strip()
    if not text:
        return None

    # paragraph-first strategy
    paragraphs = split_paragraphs(text)
    candidate_paragraphs = []

    for p in paragraphs:
        wc = len(p.split())
        if min_words <= wc <= max_words:
            candidate_paragraphs.append(p)

    if candidate_paragraphs:
        best = max(candidate_paragraphs, key=score_passage)
        return normalize_text(best)

    # if paragraphs too long/short, fallback to sentence windows
    sentences = split_sentences(text)
    if not sentences:
        return None

    windows = []
    n = len(sentences)

    # try windows of 3-6 sentences
    for window_size in range(3, 7):
        for i in range(n - window_size + 1):
            chunk = " ".join(sentences[i:i + window_size]).strip()
            wc = len(chunk.split())
            if min_words <= wc <= max_words:
                windows.append(chunk)

    if windows:
        best = max(windows, key=score_passage)
        return normalize_text(best)

    # last fallback: first max_words words if long enough
    words = text.split()
    if len(words) >= min_words:
        return " ".join(words[:max_words])

    return None

In [ ]:
# =========================
# 5) Balanced document sampling by topic group
# =========================
TARGET_TOTAL = 80
TOPIC_NAMES = list(topic_groups.keys())
PER_TOPIC = max(6, TARGET_TOTAL // len(TOPIC_NAMES))

samples = []

for topic in TOPIC_NAMES:
    subset = df_topics[df_topics['topic_groups'].apply(lambda groups: topic in groups)].copy()
    if subset.empty:
        continue

    # shuffle for reproducibility
    subset = subset.sample(frac=1, random_state=42)

    # oversample candidates because some passages may fail extraction
    subset = subset.head(PER_TOPIC * 3).copy()
    subset['sampled_topic'] = topic
    samples.append(subset)

candidate_docs = pd.concat(samples).drop_duplicates(subset=['title', 'content']).reset_index(drop=True)

print("Candidate docs:", candidate_docs.shape)


Candidate docs: (240, 8)


In [ ]:
# =========================
# 6) Extract one representative passage per sampled document
# =========================
candidate_docs['passage'] = candidate_docs['content'].apply(extract_best_passage)
candidate_docs['passage_word_count'] = candidate_docs['passage'].apply(lambda x: len(str(x).split()) if pd.notna(x) and x else 0)

selected = candidate_docs.dropna(subset=['passage']).copy()
selected = selected[selected['passage_word_count'] >= 60].copy()

print("Docs with extracted passages:", selected.shape)



Docs with extracted passages: (240, 10)


In [29]:
# =========================
# 7) Re-balance final selection by topic
# =========================
final_parts = []

for topic in TOPIC_NAMES:
    subset = selected[selected['sampled_topic'] == topic].copy()
    if subset.empty:
        continue

    n = min(PER_TOPIC, len(subset))
    final_parts.append(subset.head(n))

selected_passages = pd.concat(final_parts).drop_duplicates(subset=['title', 'passage']).reset_index(drop=True)

# Trim to target total if needed
if len(selected_passages) > TARGET_TOTAL:
    selected_passages = selected_passages.sample(n=TARGET_TOTAL, random_state=42).reset_index(drop=True)

# Add ids
selected_passages.insert(0, 'passage_id', [f"p_{i:03d}" for i in range(1, len(selected_passages) + 1)])

# Keep useful columns
keep_cols = [
    'passage_id',
    'title',
    'sampled_topic',
    'tags',
    'passage',
    'passage_word_count'
]

# include optional metadata if present
for extra_col in ['date', 'source', 'url']:
    if extra_col in selected_passages.columns:
        keep_cols.insert(keep_cols.index('tags'), extra_col)

selected_passages = selected_passages[keep_cols].copy()

print("Final selected passages:", selected_passages.shape)
print(selected_passages['sampled_topic'].value_counts())
print(selected_passages[['passage_id', 'sampled_topic', 'title', 'passage_word_count']].head())


Final selected passages: (80, 8)
sampled_topic
generative_ai         10
nlp                   10
robotics_autonomy     10
ethics_governance     10
technical_ai          10
reasoning_planning    10
applications          10
emerging_tech         10
Name: count, dtype: int64
  passage_id  sampled_topic  \
0      p_001  generative_ai   
1      p_002  generative_ai   
2      p_003  generative_ai   
3      p_004  generative_ai   
4      p_005  generative_ai   

                                               title  passage_word_count  
0  swe agents too cheap to meter the token data w...                 220  
1  running ai models without gpus on serverless p...                 220  
2    can platform engineering accelerate ai adoption                 220  
3                         20 top companies in geneva                 220  
4  google cloud next the 10 biggest google produc...                 220  


In [28]:
# =========================
# 8) Save result
# =========================
if ENV == "colab":
    save_path = '/content/drive/My Drive/CLT/stage_3/selected_passages.csv'
else:
    save_path = 'data/stage_3/selected_passages.csv'

selected_passages.to_csv(save_path, index=False, encoding='utf-8')
print("Saved to:", save_path)


# =========================
# 9) Optional: inspect a few passages
# =========================
for _, row in selected_passages.head(3).iterrows():
    print("\n" + "="*80)
    print("PASSAGE ID:", row['passage_id'])
    print("TOPIC:", row['sampled_topic'])
    print("TITLE:", row['title'])
    print("TAGS:", row['tags'])
    print("PASSAGE:", row['passage'][:800], "...")

Saved to: /content/drive/My Drive/CLT/stage_3/selected_passages.csv

PASSAGE ID: p_001
TOPIC: generative_ai
TITLE: swe agents too cheap to meter the token data war and the rise of tiny teams
TAGS: ['copilot', 'gpt', 'openai', 'claude', 'syntheticdata', 'robotics', 'cohere', 'uber', 'vibecoding', 'bolt']
PASSAGE: the recent launches of the codex and jules devin like autonomous coding agents came with some interesting pricing decisions they come at no extra cost1 you no longer have a reason to send off every thought to the agents even more than once even if you intend to eventually code it yourself let me repeat you can fire off a codex request every 60 seconds and they don t charge you extra you should use this and of course there are a lots of premium and open source coding agents with bring your own key and reasonably metered usage like github copilot coding agent and the newer amp code without normalizing for the macro base rate and the shift in job definitions however our sense is t

## 2. Generate 200 questions

In [32]:
QUESTIONS_PER_PASSAGE = 3
if ENV == "colab":
    path = '/content/drive/My Drive/CLT/stage_3/selected_passages.csv'
else:
    path = 'data/stage_3/selected_passages.csv'

df = pd.read_csv(path)

print(df.shape)
print(df.head(2))

def build_prompt(passage, questions_per_passage=3):
    return f"""
Generate exactly {questions_per_passage} question-answer pairs from the passage below.

Rules:
- Return ONLY valid JSON
- Do not use markdown
- Do not add comments or explanations
- Output must be a JSON array
- Each item must have keys: question, answer, type
- type must be one of: factual, analytical, comparative
- Questions must be answerable using only the passage

Example format:
[
  {{
    "question": "What is ...?",
    "answer": "....",
    "type": "factual"
  }}
]

Passage:
{passage}
"""

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

qa_rows = []

for i, row in df.iterrows():
    passage = row['passage']
    passage_id = row['passage_id']

    prompt = build_prompt(passage)

    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You generate only valid JSON. Do not add markdown fences or explanations."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
        )

        output_text = response.choices[0].message.content

        # Parse JSON
        qa_list = json.loads(output_text)

        for qa in qa_list:
            qa_rows.append({
                "passage_id": passage_id,
                "question": qa["question"],
                "answer": qa["answer"],
                "type": qa["type"],
                "passage": passage
            })

        time.sleep(0.5)  # avoid rate limits

    except Exception as e:
        print(f"Error at row {i}: {e}")
        continue



(80, 8)
  passage_id                                              title  \
0      p_001  swe agents too cheap to meter the token data w...   
1      p_002  running ai models without gpus on serverless p...   

   sampled_topic      date                                                url  \
0  generative_ai  24.05.25          https://www.latent.space/p/token-data-war   
1  generative_ai  25.11.24  https://thenewstack.io/running-ai-models-witho...   

                                                tags  \
0  ['copilot', 'gpt', 'openai', 'claude', 'synthe...   
1  ['llama', 'largelanguagemodel', 'metaai', 'nat...   

                                             passage  passage_word_count  
0  the recent launches of the codex and jules dev...                 220  
1  we re so glad you re here you can expect all t...                 220  

--- RAW OUTPUT row 0 ---
'[\n  {\n    "question": "What is the pricing decision for the codex and jules devin coding agents?",\n    "answer": "They com

In [33]:
# Step 5 — Create Q&A dataset
qa_df = pd.DataFrame(qa_rows)

print("Generated QAs:", qa_df.shape)
print(qa_df.head())

Generated QAs: (240, 5)
  passage_id                                           question  \
0      p_001  What is the pricing decision for the codex and...   
1      p_001  How frequently can you send a codex request wi...   
2      p_001  How has the competitive landscape for coding a...   
3      p_002              What days can you expect TNS content?   
4      p_002  How do GPUs compare to CPUs in machine learnin...   

                                              answer         type  \
0                        They come at no extra cost.      factual   
1                                  Every 60 seconds.      factual   
2  The norm of charging a premium over token usag...   analytical   
3                              Monday through Friday      factual   
4  GPUs are the preferred choice for machine lear...  comparative   

                                             passage  
0  the recent launches of the codex and jules dev...  
1  the recent launches of the codex and jules de

In [34]:
# Step 6 — Clean & limit to ~200
qa_df = qa_df.drop_duplicates(subset=['question'])

# limit to 200–300
qa_df = qa_df.head(250).reset_index(drop=True)

print("Final QAs:", qa_df.shape)

Final QAs: (238, 5)


In [35]:
if ENV == "colab":
    save_path = '/content/drive/My Drive/CLT/stage_3/qa_dataset.csv'
else:
    save_path = 'data/stage_3/qa_dataset.csv'

qa_df.to_csv(save_path, index=False)

print("Saved to:", save_path)

Saved to: /content/drive/My Drive/CLT/stage_3/qa_dataset.csv
